In [ ]:
import json, os, time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import constants

In [ ]:
HISTORY_URL = "https://api.isthereanydeal.com/games/history/v2"
CACHE_DIR = "../../data/raw/itad/price_history"
OUTPUT_PATH = "../../data/processed/game_history.parquet"

game_ids = (
    pd.read_parquet("../../data/interim/game_list.parquet")["itad_uuid"]
    .dropna()
    .unique()
    .tolist()
)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
def retry_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    return session


def fetch_history(game_id, session):
    cache_path = os.path.join(CACHE_DIR, f"{game_id}.json")
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as file:
            return json.load(file), True

    response = session.get(
        HISTORY_URL,
        params={
            "key": constants.API_KEY,
            "id": game_id,
            "country": "US",
            "shops": 61,
            "since": "2000-01-01T00:00:00Z",
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    with open(cache_path, "w", encoding="utf-8") as file:
        json.dump(data, file)
    return data, False

In [ ]:
def collect_history(game_ids, rpm=60):
    session = retry_session()
    rows = []

    for index, game_id in enumerate(game_ids, start=1):
        data, cached = fetch_history(game_id, session)
        for entry in data:
            deal = entry["deal"]
            if entry["shop"]["id"] != 61 or deal["price"]["currency"] != "USD":
                raise ValueError("Price history contains a non-Steam or non-USD record")
            rows.append({
                "itad_uuid": game_id,
                "timestamp": entry["timestamp"],
                "deal_price": deal["price"]["amount"],
                "regular_price": deal["regular"]["amount"],
                "percent": deal["cut"],
            })
        if not cached:
            time.sleep(60 / rpm)
        if index % 500 == 0:
            print(f"Processed {index:,} of {len(game_ids):,} games")

    history = pd.DataFrame(rows).drop_duplicates()
    history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)
    history["deal_price"] = history["deal_price"].astype("float32")
    history["regular_price"] = history["regular_price"].astype("float32")
    history["percent"] = history["percent"].astype("uint8")
    return history.sort_values(["itad_uuid", "timestamp"]).reset_index(drop=True)

In [ ]:
game_history = collect_history(game_ids)
game_history.to_parquet(OUTPUT_PATH, index=False)
print(f"Wrote {len(game_history):,} rows to {OUTPUT_PATH}")